# Tools

In [17]:
import os 
import json 
import yfinance as yf
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import display, Markdown

load_dotenv()  # Load environment variables from .env file
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))  # Initialize OpenAI

In [3]:
def get_stock_data(ticker: str):
    """
    Fetch real stock data from yahoo Finance.
    Returns price, P/E ratio, market cap, and 52-week range
    """

    stock = yf.Ticker(ticker)
    info = stock.info

    data = {
        "ticker": ticker,
        "current_price": info.get("currentPrice"),
        "pe_ratio": info.get("trailingPE"),
        "market_cap": info.get("marketCap"),
        "52_week_high": info.get("fiftyTwoWeekHigh"),
        "52_week_low": info.get("fiftyTwoWeekLow"),
        "revenue_growth": info.get("revenueGrowth"),
        "profit_margin": info.get("profitMargins")
    }

    return data

In [4]:
nvda_data = get_stock_data("NVDA")
print(json.dumps(nvda_data, indent=4))

{
    "ticker": "NVDA",
    "current_price": 207.56,
    "pe_ratio": 31.834356,
    "market_cap": 5027310600192,
    "52_week_high": 236.54,
    "52_week_low": 138.83,
    "revenue_growth": 0.852,
    "profit_margin": 0.62966
}


In [5]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_stock_data",
            "description": "Fetch real stock data from yahoo Finance. Returns price, P/E ratio, market cap, and 52-week range",
            "parameters": {
                "type": "object",
                "properties": {
                    "ticker": {
                        "type": "string",
                        "description": "The stock ticker symbol (e.g., AAPL for Apple Inc.)"
                    }
                },
                "required": ["ticker"]
            }
        }
    }
]

In [6]:
messages = [
    {"role": "system", "content" : "You are a finantial research assistant use tools to get real data befor making any claims"},
    {"role": "user", "content": "Should I invest in NVIDIA right now? Give me 3 bullet points."}
]

response = client.chat.completions.create(
    model="gpt-4o",
    messages=messages,
    tools=tools,
    tool_choice="auto"
)

stop_reason = response.choices[0].finish_reason

print(f"Model stopped because: {stop_reason}")

print(f"Response: {response.choices[0].message.content}")

Model stopped because: tool_calls
Response: None


In [8]:
tool_calls = response.choices[0].message.tool_calls
tool_calls

[ChatCompletionMessageFunctionToolCall(id='call_7ZC4ZwHRiXIbE06tnI5WWYIG', function=Function(arguments='{"ticker":"NVDA"}', name='get_stock_data'), type='function')]

In [10]:
import json 

tool_call = response.choices[0].message.tool_calls[0]
function_name = tool_call.function.name
function_args = json.loads(tool_call.function.arguments)

In [11]:
print(f"Model Wants to call function: {function_name} with arguments: {function_args}")

Model Wants to call function: get_stock_data with arguments: {'ticker': 'NVDA'}


In [12]:
if function_name == "get_stock_data":
    tool_result = get_stock_data(**function_args)

print(f"Tool Result: {tool_result}")

Tool Result: {'ticker': 'NVDA', 'current_price': 208.38, 'pe_ratio': 31.96319, 'market_cap': 5047656120320, '52_week_high': 236.54, '52_week_low': 138.83, 'revenue_growth': 0.852, 'profit_margin': 0.62966}


In [13]:
response.choices[0].message,

(ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_7ZC4ZwHRiXIbE06tnI5WWYIG', function=Function(arguments='{"ticker":"NVDA"}', name='get_stock_data'), type='function')]),)

In [14]:
message_with_result = messages + [
    response.choices[0].message,
    {"role": "tool",
    "tool_call_id": tool_call.id,
    "content": json.dumps(tool_result)}
]

In [15]:
message_with_result

[{'role': 'system',
  'content': 'You are a finantial research assistant use tools to get real data befor making any claims'},
 {'role': 'user',
  'content': 'Should I invest in NVIDIA right now? Give me 3 bullet points.'},
 ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_7ZC4ZwHRiXIbE06tnI5WWYIG', function=Function(arguments='{"ticker":"NVDA"}', name='get_stock_data'), type='function')]),
 {'role': 'tool',
  'tool_call_id': 'call_7ZC4ZwHRiXIbE06tnI5WWYIG',
  'content': '{"ticker": "NVDA", "current_price": 208.38, "pe_ratio": 31.96319, "market_cap": 5047656120320, "52_week_high": 236.54, "52_week_low": 138.83, "revenue_growth": 0.852, "profit_margin": 0.62966}'}]

In [16]:
final_response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=message_with_result,
    tools=tools,
    tool_choice="none"
)

In [18]:
final_answer = final_response.choices[0].message.content
display(Markdown(final_answer))

Here are three bullet points to consider regarding investing in NVIDIA (NVDA):

1. **Current Valuation**: NVIDIA is trading at approximately $208.38 with a P/E ratio of about 31.96, indicating it may be relatively high compared to the overall market, depending on growth expectations.

2. **Growth Potential**: The company has shown impressive revenue growth of 85.2% and maintains a solid profit margin of 62.97%. This suggests strong business fundamentals and potential for future growth, particularly in sectors like gaming and AI.

3. **Market Position**: With a market capitalization of approximately $5.05 trillion, NVIDIA is a dominant player in the semiconductor industry. However, it is essential to consider its 52-week range, with a high of $236.54 and a low of $138.83, indicating potential volatility in its stock price.

Make sure to assess your investment strategy and risk tolerance before making a decision.